# 07b — Breakout Stop-loss And Regime Validation

Notebook này gom phần nghiên cứu breakout vào một flow ngắn hơn:
- chuẩn hóa feature cần cho breakout research
- backtest non-overlap và quét fixed stop / ATR stop
- tạo setup eligibility và volatility rule để notebook trade card dùng lại


In [1]:
# Cell: Setup Project Root
import os
import sys
from pathlib import Path


def find_project_root(start_path=None):
    path = Path(start_path or os.getcwd()).resolve()
    for parent in [path, *path.parents]:
        if (parent / "src").is_dir() and (parent / "config").is_dir():
            return parent
    raise FileNotFoundError("Khong tim thay project root co thu muc src/ va config/")


project_root = find_project_root()
os.chdir(project_root)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print("Project root:", project_root)


Project root: /Users/jamesb/Desktop/vietnam-quant-research


In [2]:
# Cell: Import Breakout Research Modules
import pandas as pd

from src.backtest.breakout_research import run_breakout_research, save_breakout_research_outputs
from src.data.loaders import read_parquet


In [3]:
# Cell: Load Feature Frame
candidate_paths = [
    "data/signals/setup_signals.parquet",
    "data/processed/stock_features.parquet",
    "data/processed/stock_features_full.parquet",
    "data/processed/clean_daily_ohlcv.parquet",
]

features = pd.DataFrame()
loaded_from = None

for candidate_path in candidate_paths:
    if not Path(candidate_path).exists():
        continue
    candidate_df = read_parquet(candidate_path)
    if candidate_df.empty:
        continue
    features = candidate_df.copy()
    loaded_from = candidate_path
    break

if features.empty:
    raise FileNotFoundError(
        "Khong tim thay data dau vao cho notebook 07b. "
        "Hay tao du lieu truoc bang scripts/generate_features.py va scripts/generate_signals.py"
    )

if "date" not in features.columns:
    raise KeyError(
        f"File {loaded_from} da duoc doc nhung khong co cot 'date'. "
        f"Current columns: {features.columns.tolist()}"
    )

features["date"] = pd.to_datetime(features["date"])
print("Loaded features from:", loaded_from)
features.shape


Loaded features from: data/signals/setup_signals.parquet


(720025, 52)

In [4]:
# Cell: Run Breakout Research
results = run_breakout_research(
    features,
    signal_col="breakout_20d",
    holding_period=20,
    cost=0.004,
    stop_losses=[None, -0.05, -0.07, -0.10, -0.12],
    atr_multiples=[1.5, 2.0, 2.5],
)

pd.DataFrame([results["baseline_summary"]])


,trades,win_rate,avg_return,median_return,best_return,worst_return,gross_profit,gross_loss,profit_factor,stop_rate,entry_basis
0,8218,0.478219,0.014741,-0.004,2.938598,-0.580271,491.227549,370.085937,1.327334,0.0,next_open


In [5]:
# Cell: Review Fixed Stop Results
display(results["fixed_stop_summary"])
display(pd.DataFrame([results["best_stop_row"]]))
display(results["by_symbol"].head(10))
display(results["by_volatility_regime"])


,stop_loss,trades,win_rate,avg_return,median_return,best_return,worst_return,gross_profit,gross_loss,profit_factor,stop_rate
0,no_stop,5590,0.516637,0.022030,0.004064,2.39618,-0.580271,360.773271,237.623045,1.518259,0.000000
1,-0.12,5590,0.488909,0.017525,-0.004000,2.39618,-0.500900,339.769098,241.803956,1.405143,0.230054
2,-0.1,5590,0.471020,0.016458,-0.009355,2.39618,-0.310533,328.592927,236.591666,1.388861,0.296422
3,-0.07,5590,0.416458,0.013867,-0.038634,2.39618,-0.310533,298.726546,221.209914,1.350421,0.441503
4,-0.05,5590,0.341860,0.010075,-0.054000,1.57809,-0.328204,254.757968,198.437973,1.283817,0.570841


,stop_loss,trades,win_rate,avg_return,median_return,best_return,worst_return,gross_profit,gross_loss,profit_factor,stop_rate
0,no_stop,5590,0.516637,0.02203,0.004064,2.39618,-0.580271,360.773271,237.623045,1.518259,0.0


,symbol,trades,win_rate,avg_return,median_return,best_return,worst_return,gross_profit,gross_loss,profit_factor,stop_rate
0,TCM,1,1.0,0.013912,0.013912,0.013912,0.013912,0.013912,0.0,inf,0.0
1,VNS,1,1.0,0.027579,0.027579,0.027579,0.027579,0.027579,0.0,inf,0.0
2,QNP,2,1.0,0.080083,0.080083,0.088767,0.071398,0.160165,0.0,inf,0.0
3,VFG,2,1.0,0.177139,0.177139,0.209044,0.145233,0.354277,0.0,inf,0.0
4,RAL,1,1.0,0.009797,0.009797,0.009797,0.009797,0.009797,0.0,inf,0.0
5,MDG,1,1.0,0.541455,0.541455,0.541455,0.541455,0.541455,0.0,inf,0.0
6,CLW,1,1.0,0.012639,0.012639,0.012639,0.012639,0.012639,0.0,inf,0.0
7,CVT,1,1.0,0.068797,0.068797,0.068797,0.068797,0.068797,0.0,inf,0.0
8,TTE,2,1.0,0.394244,0.394244,0.494911,0.293578,0.788489,0.0,inf,0.0
9,ADC,2,1.0,0.038220,0.038220,0.056617,0.019823,0.076439,0.0,inf,0.0


,volatility_regime,trades,win_rate,avg_return,median_return,best_return,worst_return,gross_profit,gross_loss,profit_factor,stop_rate
0,Low Vol,323,0.439628,0.015287,-0.009490,1.401063,-0.399931,15.846132,10.908589,1.452629,0.0
1,Unknown,21,0.523810,0.003985,0.002667,0.373368,-0.241079,1.139247,1.055552,1.079290,0.0
2,Normal Vol,672,0.397321,-0.000583,-0.019348,1.961517,-0.423753,27.688051,28.079852,0.986047,0.0
3,High Vol,693,0.396825,-0.002943,-0.026063,2.938598,-0.546383,33.100577,35.140122,0.941960,0.0
4,Stress Vol,421,0.375297,-0.003714,-0.022227,0.955943,-0.515211,20.445093,22.008625,0.928958,0.0
5,Panic Vol,265,0.343396,-0.024012,-0.043130,0.820601,-0.475353,11.772233,18.135480,0.649127,0.0


In [6]:
# Cell: Review ATR And Rules
display(results["atr_stop_summary"])
display(results["eligibility"])
display(results["volatility_rule"])


,atr_multiple,trades,win_rate,avg_return,median_return,best_return,worst_return,gross_profit,gross_loss,profit_factor,stop_rate
0,2.5,5590,0.423256,0.013421,-0.031918,2.39618,-0.310533,315.398371,240.377220,1.312098,0.0
1,2.0,5590,0.377639,0.011418,-0.040736,2.39618,-0.342183,289.782486,225.953656,1.282486,0.0
2,1.5,5590,0.317352,0.010116,-0.038051,2.39618,-0.310533,257.162717,200.614340,1.281876,0.0


,symbol,setup_name,trades,win_rate,avg_return,median_return,profit_factor,stop_rate,edge_tier,eligible,watchlist_eligible,stop_loss,holding_period,cost_assumption
0,LPB,breakout_20d,10,0.70,0.047243,0.076296,3.989008,0.0,Valid Edge,True,True,-0.1,20,0.004
1,SBT,breakout_20d,10,0.60,0.041818,0.041770,3.578414,0.0,Valid Edge,True,True,-0.1,20,0.004
2,MZG,breakout_20d,5,0.80,0.131775,0.106169,35.403185,0.0,Emerging Edge,False,True,-0.1,20,0.004
3,TVN,breakout_20d,8,0.75,0.073752,0.078399,32.656184,0.0,Emerging Edge,False,True,-0.1,20,0.004
4,CDC,breakout_20d,5,0.80,0.161575,0.105392,24.869166,0.0,Emerging Edge,False,True,-0.1,20,0.004
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
468,RYG,breakout_20d,2,0.00,-0.124581,-0.124581,0.000000,0.0,Insufficient Data,False,False,-0.1,20,0.004
469,TVC,breakout_20d,5,0.00,-0.075157,-0.055282,0.000000,0.0,Insufficient Data,False,False,-0.1,20,0.004
470,SBA,breakout_20d,3,0.00,-0.021140,-0.007367,0.000000,0.0,Insufficient Data,False,False,-0.1,20,0.004
471,SCS,breakout_20d,2,0.00,-0.051888,-0.051888,0.000000,0.0,Insufficient Data,False,False,-0.1,20,0.004


,volatility_regime,setup_name,trades,win_rate,avg_return,median_return,profit_factor,stop_rate,volatility_rule,stop_loss,holding_period
0,Low Vol,breakout_20d,323,0.439628,0.015287,-0.009490,1.452629,0.0,Conditional,-0.1,20
1,Unknown,breakout_20d,21,0.523810,0.003985,0.002667,1.079290,0.0,Ignore,-0.1,20
2,Normal Vol,breakout_20d,672,0.397321,-0.000583,-0.019348,0.986047,0.0,Avoid,-0.1,20
3,High Vol,breakout_20d,693,0.396825,-0.002943,-0.026063,0.941960,0.0,Avoid,-0.1,20
4,Stress Vol,breakout_20d,421,0.375297,-0.003714,-0.022227,0.928958,0.0,Avoid,-0.1,20
5,Panic Vol,breakout_20d,265,0.343396,-0.024012,-0.043130,0.649127,0.0,Avoid,-0.1,20


In [7]:
# Cell: Save Breakout Outputs
save_breakout_research_outputs(results, out_dir="data/signals", prefix="breakout_20d")
print("Saved breakout research artifacts to data/signals")


Saved breakout research artifacts to data/signals
